In [1]:
    !pip install -U datasets huggingface_hub weaviate-client

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 828.9 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 2.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 4.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 11.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 652.7/652.7 kB 9.2 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 6.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 7.1 MB/s

In [2]:
import sys
sys.path.insert(0, "/home/jovyan")

from datasets import load_dataset
from weaviate.classes.config import Property, DataType
from common import TEXT2VEC_CONFIG, connect

if 'client' in globals():
    client.close()

client = connect()

print("Weaviate готов к работе:", client.is_ready())

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Weaviate готов к работе: True


In [3]:
import sys
sys.path.insert(0, "/home/jovyan")

from common import load_env

env_path = load_env()
print(f"HF_TOKEN: {'задан' if __import__('os').getenv('HF_TOKEN') else 'не задан'}")
if env_path:
    print(f"Загружен .env: {env_path}")

HF_TOKEN: задан
Загружен .env: /home/jovyan/examples/.env


In [4]:
print("Качаем датасет AG News...")
dataset = load_dataset("SetFit/ag_news", split="train")
dataset = dataset.select(range(1000))
category_map = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

Качаем датасет AG News...


Generating test split: 100%|██████████| 7600/7600 [00:00<00:00, 275647.56 examples/s]


In [5]:
try:
    client.collections.delete("News")
    print("Deleted existing News collection")
except:
    print("No existing collection")

Deleted existing News collection


In [6]:
news_collection = client.collections.create(
    name="News",
    vector_config=TEXT2VEC_CONFIG,
    properties=[
        Property(name="title", data_type=DataType.TEXT),
        Property(name="description", data_type=DataType.TEXT),
        Property(name="category", data_type=DataType.TEXT),
    ]
)
print("Collection created!")

Collection created!


In [7]:
weaviate_data = [
    {
        "title": row["text"].split(" - ")[0] if " - " in row["text"] else "News",
        "description": row["text"],
        "category": category_map[row["label"]]  # Using category_map
    }
    for row in dataset 
]

print(f"Prepared {len(weaviate_data)} records")
print("Sample record:", weaviate_data[0])

Prepared 1000 records
Sample record: {'title': 'Wall St. Bears Claw Back Into the Black (Reuters) Reuters', 'description': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'category': 'Business'}


In [8]:
import time

In [9]:
print("Загружаем данные в Weaviate...")
start_time = time.time()

with news_collection.batch.fixed_size(batch_size=20, concurrent_requests=2) as batch:
    for i, record in enumerate(weaviate_data):
        batch.add_object(
            properties=record
        )
        if i % 100 == 0:
            print(f"Загружено {i} записей...")

print(f"Загружено {len(weaviate_data)} записей за {time.time() - start_time:.2f} сек")

Загружаем данные в Weaviate...
Загружено 0 записей...
Загружено 100 записей...
Загружено 200 записей...
Загружено 300 записей...
Загружено 400 записей...
Загружено 500 записей...
Загружено 600 записей...
Загружено 700 записей...
Загружено 800 записей...
Загружено 900 записей...
Загружено 1000 записей за 191.11 сек


In [14]:
import sys
sys.path.insert(0, "/home/jovyan")

from common import SEARCH_METADATA_VECTOR, display_objects

def semantic_search(query, limit=100):
    """Поиск по смыслу без фильтров"""
    return news_collection.query.near_text(
        query=query,
        limit=limit,
        return_properties=["title", "description", "category"],
        return_metadata=SEARCH_METADATA_VECTOR
    )

print("Семантический поиск: 'technology breakthrough innovation'")
semantic_results = semantic_search("technology breakthrough innovation")
display_objects(semantic_results.objects, title="СЕМАНТИЧЕСКИЙ ПОИСК", max_text=250, search_type="vector")

Семантический поиск: 'technology breakthrough innovation'

=== СЕМАНТИЧЕСКИЙ ПОИСК ===

#1  [cosine distance=0.5282 | cosine similarity=0.4718 (47.2%) | certainty=0.7359 (73.6%) | хорошее соответствие]
  title: Nanotech Research Spending Seen Reaching \$8.6 Bln  SAN FRANCISCO (Reuters)
  description: Nanotech Research Spending Seen Reaching \$8.6 Bln  SAN FRANCISCO (Reuters) - Worldwide research and  development spending in the emerging field of nanotechnology  should rise about 10 percent this year to \$8.6 billion, a  research firm said on Mond...
  category: Sci/Tech

#2  [cosine distance=0.5420 | cosine similarity=0.4580 (45.8%) | certainty=0.7290 (72.9%) | хорошее соответствие]
  title: News
  description: Locusts Inspire Technology That May Prevent Car Crashes Locusts are commonly associated with plagues, food shortages, and death. But they are also inspiring what may be the next wave in lifesaving collision-avoidance systems.
  category: Sci/Tech

#3  [cosine distance=0.5434 | c

In [15]:
@client.close()
print("Соединение закрыто")

Соединение закрыто
